In [ ]:
import gc
import sys
from pathlib import Path

# Ensure project root is in sys.path
root_dir = Path.cwd().resolve()
for candidate in (root_dir, *root_dir.parents):
    if (candidate / ".oswald-root").exists():
        root_dir = candidate
        break
    if (candidate / "terrain_diffusion").is_dir() and (candidate / "oswald").is_dir():
        root_dir = candidate
        break
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

# Set PyTorch expandable segments
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clean stale models, tensors, and cached memory from VRAM if re-running
for _k in list(globals().keys()):
    _v = globals().get(_k)
    if _v is not None and (
        hasattr(_v, "to") or hasattr(_v, "parameters") or _k in ("elev_map", "pipe", "pipeline")
    ):
        try:
            del globals()[_k]
        except Exception:
            pass

if hasattr(sys, "last_traceback"):
    sys.last_traceback = None
    sys.last_value = None
    sys.last_type = None

gc.collect()

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Check NVIDIA hardware access and VRAM availability
assert torch.cuda.is_available(), "NVIDIA GPU hardware is not accessible via PyTorch CUDA runtime."

device_idx = torch.cuda.current_device()
device_name = torch.cuda.get_device_name(device_idx)
free_bytes, total_bytes = torch.cuda.mem_get_info(device_idx)
free_gb = free_bytes / (1024 ** 3)
total_gb = total_bytes / (1024 ** 3)

print(f"NVIDIA GPU: {device_name} (Device {device_idx})")
print(f"VRAM: {free_gb:.2f} GB free / {total_gb:.2f} GB total")

# Threshold check (recommended ~2.0 GB minimum free memory for generation)
MIN_VRAM_GB = 2.0
if free_gb < MIN_VRAM_GB:
    print(f"[WARN] Free VRAM ({free_gb:.2f} GB) is below the recommended {MIN_VRAM_GB:.1f} GB. Other GPU processes may cause OOM.")
else:
    print("[PASS] GPU hardware accessible with sufficient VRAM.")

import ast
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from oswald.paths import get_project_path

with open(get_project_path("legends/topography.txt"), "r") as data:
    cdict = ast.literal_eval(data.read())
    topo = LinearSegmentedColormap('topo', cdict)

In [ ]:
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from pathlib import Path

# folder containing the source data
source_folder = "working/92757802"

# fetch the source data from the folder and scale elevations to suit terrain_diffusion input
img = Image.open(Path.joinpath(Path(get_project_path(source_folder)), "heightfield.tif"))
arr = np.array(img)
arr *= 4096
arr -= 1024

world_coarse_map = Image.fromarray(arr)

In [ ]:
import globe_viewer as gv

globe_viewer = gv.GlobeViewer(arr, cmap=topo, display_size=800)
globe_viewer.show()

#globe_viewer.set_relief_shading(True)
#globe_viewer.set_relief_intensity(2.0)
#globe_viewer.refresh()

In [ ]:
import oswald.coarse_map as cm
import oswald.map_viewer as mv

lat, lon = globe_viewer.centre()
print (lat, lon)
size = 64
extent = 750

mapper = cm.CoarseMap (size=size, extent=extent)
mapper.add_channel (coarse_map = world_coarse_map, channel = "heightmap")
cond_map = mapper.get_cond_map (lat = lat, lon = lon)

cond_viewer = mv.MapViewer(cond_map, cmap=topo, display_size=800)
cond_viewer.show()

cond_viewer.set_relief_shading(True)
cond_viewer.set_relief_intensity(2.0)
cond_viewer.refresh()


In [ ]:
from oswald.generate_map import MapGenerator

generator = MapGenerator(
    device="cuda",
    seed=42,
    snr="0.1,0.2,1.0,0.2,1.0",
    batch_size=2,
    dtype= "fp32",
    torch_compile=False,
    caching_strategy="direct")
elev_map = generator.generate(cond_map)
generator.close()  # frees VRAM

print ("Done.")

In [ ]:
import importlib
import oswald.map_viewer as viewer_module
importlib.reload(viewer_module)

out_viewer = viewer_module.MapViewer(elev_map, cmap=topo, display_size=800)
out_viewer.show()

out_viewer.set_relief_shading(True)
out_viewer.set_relief_intensity(2.0)
out_viewer.refresh()
